# Analyze results for citation optimization

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
from IPython.display import display

In [ ]:
pd.options.plotting.backend = "plotly"

create_images = False

baseline_runs = [
    "", "", "" # enter results files to analyse
]

comparing_runs = [
    "", "", "" # enter results files to analyse
]

metrics = [
    "context_precision","context_recall","nv_context_relevance","answer_relevancy",
    "nv_response_groundedness", "correctness","semantic_similarity",
    "trustworthiness","prompt_alignment"
]

metrics_to_compare = [
    "trustworthiness", "semantic_similarity", "prompt_alignment", "nv_response_groundedness"
]

Flatten and combine all results from all runs into one dataframe

In [ ]:
baseline_data = []

for run_file in baseline_runs:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                baseline_data.append(r)

baseline_df = pd.DataFrame(baseline_data)

baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
comparing_data = []

for run_file in comparing_runs:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                comparing_data.append(r)

comparing_df = pd.DataFrame(comparing_data)

comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
baseline_detail = baseline_df.groupby('dataset')[metrics].mean()
baseline_by_metric = baseline_df[metrics_to_compare].mean()

comparing_detail = comparing_df.groupby('dataset')[metrics].mean()
comparing_by_metric = comparing_df[metrics_to_compare].mean()

baseline_by_dataset = baseline_df.groupby('dataset')[metrics_to_compare].mean().T.mean()
comparing_by_dataset = comparing_df.groupby('dataset')[metrics_to_compare].mean().T.mean()

baseline_by_run = baseline_df.groupby('run')[metrics_to_compare].mean()
comparing_by_run = comparing_df.groupby('run')[metrics_to_compare].mean()

## Visualizing results

In [ ]:
baseline_overall = baseline_df[metrics].mean().mean()
comparing_overall = comparing_df[metrics].mean().mean()

fig = go.Figure(go.Indicator(
    mode = "number+delta",
    value = np.round(comparing_overall,3),
    number = {'prefix': f"Overall changed from {np.round(baseline_overall,3)} to "},
    delta = {'position': "top", 'reference': np.round(baseline_overall,3)}
))

fig.show()

if create_images:
    pio.write_image(fig, 'overall-delta.pdf')

In [ ]:
fig = go.Figure()

baseline_text = np.round(baseline_by_metric.values.tolist(), 2)
comparing_text = np.round(comparing_by_metric.values.tolist(), 2)

fig.add_trace(
    go.Bar(
        x=baseline_by_metric.index.tolist(),
        y=baseline_by_metric.values.tolist(),
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=baseline_by_metric.index.tolist(),
        y=comparing_by_metric.values.tolist(),
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)

fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="Metric"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'metric-comparison-bar.pdf')

In [ ]:
fig = go.Figure()

baseline_groundedness = baseline_df['nv_response_groundedness'].mean()
comparing_groundedness = comparing_df['nv_response_groundedness'].mean()

baseline_text = np.round(baseline_groundedness, 2)
comparing_text = np.round(comparing_groundedness, 2)

fig.add_trace(
    go.Bar(
        x=["baseline"],
        y=[baseline_groundedness],
        name="baseline",
        marker_color="grey",
        text=[baseline_text]
    )
)

fig.add_trace(
    go.Bar(
        x=["comparing"],
        y=[comparing_groundedness],
        name="comparing",
        marker_color='#0EA4E3',
        text=[comparing_text]
    )
)

fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="nv_response_groundedness"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=400,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'groundedness-comparison.pdf')

In [ ]:
fig = go.Figure()

baseline_text = np.round(baseline_by_dataset.values.tolist(), 2)
comparing_text = np.round(comparing_by_dataset.values.tolist(), 2)

fig.add_trace(
    go.Bar(
        x=baseline_by_dataset.index.tolist(),
        y=baseline_by_dataset.values.tolist(),
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=baseline_by_dataset.index.tolist(),
        y=comparing_by_dataset.values.tolist(),
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)


fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="Dataset"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'dataset-comparison-bar.pdf')

In [ ]:
metrics_diffs = comparing_by_metric - baseline_by_metric
metrics_percentage_diff = np.round(metrics_diffs / baseline_by_metric * 100, 2)

fig = go.Figure()

for metric in metrics_diffs.index:
    y = metrics_diffs[metric]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[metric], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({metrics_percentage_diff[metric]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.45, 0.45],
        title="Delta"
    ),
    xaxis=dict(
        title=""
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'metrics-delta-bar.pdf')

In [ ]:
dataset_diffs = comparing_by_dataset - baseline_by_dataset
dataset_percentage_diffs = np.round(dataset_diffs / baseline_by_dataset * 100, 2)

fig = go.Figure()

for dataset in dataset_diffs.index:
    y = dataset_diffs[dataset]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[dataset], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({dataset_percentage_diffs[dataset]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.4, 0.4],
        title="Delta"
    ),
    xaxis=dict(
        title="Dataset"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'dataset-delta-bar.pdf')

In [ ]:
comparing_by_metric = comparing_df[['nv_response_groundedness', 'correctness']].mean()
baseline_by_metric = baseline_df[['nv_response_groundedness', 'correctness']].mean()

metrics_diffs = comparing_by_metric-baseline_by_metric

metrics_percentage_diff = np.round(metrics_diffs / baseline_by_metric * 100, 2)

fig = go.Figure()

for metric in metrics_diffs.index:
    y = metrics_diffs[metric]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[metric], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({metrics_percentage_diff[metric]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.4, 0.4],
        title="Delta"
    ),
    xaxis=dict(
        title=""
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

Zusätzlich zu den erwarteten Veränderungen zeigte zum einen die `nv_response_groundedness` einen deutlichen Ansteig. Die `correctness` sank leicht, der Grund dafür wird im Folgenden untersucht:

In [ ]:
responses = comparing_df.loc[comparing_df['dataset'] == 'fact checking']['response']
references = comparing_df.loc[comparing_df['dataset'] == 'fact checking']['reference']
cleaned_responses = [res.split("<bib>")[0] for res in responses]
correctness_scores = comparing_df.loc[comparing_df['dataset'] == 'fact checking']['correctness'].tolist()

true_and_fully_correct = []
true_and_not_fully_correct = []
true_answers = 0
all_ansers = 0
for i, ref in enumerate(references):
    all_ansers += 1
    if ref == cleaned_responses[i][0]:
        if correctness_scores[i] < 1:
            true_and_not_fully_correct.append(responses.tolist()[i])
        else:
            true_and_fully_correct.append(responses.tolist()[i])
        true_answers += 1

print(f"mean correctnes score: {comparing_df.loc[comparing_df['dataset'] == 'fact checking']['correctness'].mean()}")
print(f"actual mean correctness: {true_answers / all_ansers}")

print(f"true answers with correctness = 1: {true_and_fully_correct}")
print(f"true answers with correctness < 1: {true_and_not_fully_correct}")

Es ist erkennbar, dass das Quellenverzeichnis nicht nur die `semantic_similarity` und das `prompt_alignment` beeinflusst, wie bereits zuvor angenommen, sondern auch Auswirkungen auf die `correctness` hat. Diese Verschlechterung spiegelt also keinen tatsächlichen Korrektheitsrückgang wieder.

In [ ]:
detail = comparing_df.groupby('dataset')[metrics].mean()

text_matrix = detail.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

deatil_fig = go.Figure(data=go.Heatmap(
    z=detail,
    x=detail.columns,
    y=detail.index,
    text=text_matrix,
    zmin=0.0,
    zmax=1.0,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#FF7B52"), 
                (0.5, "#FFC44F"), (0.7, "#FFC44F"),
                (0.7, "#F1FF53"), (0.9, "#F1FF53"),
                (0.9, "#56D752"),  (1.00, "#56D752")]
))

deatil_fig.update_layout(
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    title="Mean score per dataset and metric",
    xaxis_nticks=len(detail.columns),
    yaxis_nticks=len(detail.index),
    font=dict(
        size=16,
    ),

)
    
deatil_fig.show()

if create_images:
    pio.write_image(deatil_fig, 'detail-scores.png', scale=5)

In [ ]:
heatmap_data = comparing_detail - baseline_detail

display(baseline_detail)
display(comparing_detail)

text_matrix = heatmap_data.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    text=text_matrix,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#F1FF53"), (1.00, "#56D752")],
))

fig.update_layout(
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    title="Mean delta per dataset and metric",
    xaxis_nticks=len(detail.columns),
    yaxis_nticks=len(detail.index),
    font=dict(
        size=16,
    ),

)

fig.show()

if create_images:
    pio.write_image(fig, 'detail-delta.png', scale=5)

In [ ]:
results_by_metric = comparing_df[metrics].mean()

mt_means_fig = go.Figure()

for metric in results_by_metric.index:
    y = results_by_metric[metric]
    color = "#FF7B52" if y <= 0.5 else "#FFC44F" if (y <= 0.7 and y > 0.5) else "#F1FF53" if (y <= 0.9 and y > 0.7) else "#56D752"
    mt_means_fig.add_trace(
        go.Bar(x=[metric], y=[results_by_metric[metric]], marker_color=color, text=np.round(y, 2))
    )

mt_means_fig.update_layout(
    xaxis_title='Metric',
    yaxis_title='Mean score',
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    ),
)

mt_means_fig.show()

if create_images:
    pio.write_image(mt_means_fig, 'metric-means-bar.pdf')